In [0]:
---- Databricks notebook source

DROP TABLE IF EXISTS banking.metadata.tables;
DROP TABLE IF EXISTS banking.metadata.table_parameters;
drop table if exists banking.metadata.table_watermarks;
drop table if exists banking.metadata.pipeline_runs;

In [0]:
CREATE CATALOG IF NOT EXISTS banking;

create schema if not exists banking.metadata;

In [0]:
-- Static registry of logical tables

CREATE TABLE IF NOT EXISTS banking.metadata.tables(
  table_id INT,
  table_name STRING,
  source_system STRING, -- sqlserver/ dbfs/blob
  source_schema STRING, -- dbo/ (null for blob)
  source_table STRING, -- table name (null for blob)
  source_path STRING, -- BLOB PATH (null for sql server)
  target_layer STRING, -- silver/gold
  bronze_schema STRING, -- bronze
  silver_schema STRING, -- silver
  gold_schema STRING, -- gold
  active_flag BOOLEAN,
  load_order int,
  created_at TIMESTAMP

)
USING DELTA;

In [0]:
-- processing configuration (load type, PK, watermark)

CREATE TABLE IF NOT EXISTS banking.metadata.table_parameters(
  table_id int,
  parameter_name STRING, -- load_type / Primary_key/ watermark_column
  parameter_value STRING,
  created_at TIMESTAMP
)
USING DELTA;

In [0]:
-- Stores last successful watermark per table---

create table if not exists banking.metadata.table_watermarks(
  table_id int,
  last_watermark_value string, -- flexible type storage
  last_updated_at timestamp,
  last_run_id bigint
)
using delta
partitioned by(table_id)

In [0]:
-- Execution audit table -- 
create table if not exists banking.metadata.pipeline_runs(
  run_id bigint,
  table_id int,
  layer string, -- bronze / silver/ gold
  start_time timestamp,
  end_time timestamp,
  status string, -- success /failed
  num_of_records bigint,
  error_message string
)
using DELTA
partitioned by  (table_id)


In [0]:
-- CREATE SOURCE SCHEMA

create schema if not exists banking.source;

create volume if not exists banking.source.volume;

In [0]:
INSERT INTO banking.metadata.tables VALUES
-- 1. Customers (SQL Server)
(1, 'customers', 'sqlserver', 'banking', 'customers', NULL, 'silver', 'bronze', 'silver', NULL, TRUE, 1, current_timestamp()),

-- 2. Accounts (SQL Server)
(2, 'accounts', 'sqlserver', 'banking', 'accounts', NULL, 'silver', 'bronze', 'silver', NULL, TRUE, 2, current_timestamp()),

-- 3. Transactions (SQL Server)
(3, 'transactions', 'sqlserver', 'banking', 'transactions', NULL, 'silver', 'bronze', 'silver', NULL, TRUE, 3, current_timestamp()),

-- 4. Branches (SQL Server - Full Load)
(4, 'branches', 'sqlserver', 'banking', 'branches', NULL, 'silver', 'bronze', 'silver', NULL, TRUE, 4, current_timestamp()),

-- 5. Credit Bureau Reports (Blob CSV)
(5, 'credit_bureau_reports', 'blob', NULL, NULL,
 '/Volumes/banking/source/volume/credit_bureau_reports/', 'silver',
 'bronze', 'silver', NULL, TRUE, 5, current_timestamp()),

-- 6. Payment Gateway Logs (Blob CSV)
(6, 'payment_gateway_logs', 'blob', NULL, NULL,
 '/Volumes/banking/source/volume/payment_gateway_logs/', 'silver',
 'bronze', 'silver', NULL, TRUE, 6, current_timestamp());

In [0]:
INSERT INTO banking.metadata.table_parameters VALUES

-- ================= CUSTOMERS =================
(1, 'load_type', 'MERGE', current_timestamp()),
(1, 'primary_key', 'customer_id', current_timestamp()),
(1, 'watermark_column', 'updated_at', current_timestamp()),

-- ================= ACCOUNTS =================
(2, 'load_type', 'MERGE', current_timestamp()),
(2, 'primary_key', 'account_id', current_timestamp()),
(2, 'watermark_column', 'updated_at', current_timestamp()),

-- ================= TRANSACTIONS =================
(3, 'load_type', 'APPEND', current_timestamp()),
(3, 'primary_key', 'txn_id', current_timestamp()),
(3, 'watermark_column', 'txn_timestamp', current_timestamp()),

-- ================= BRANCHES =================
(4, 'load_type', 'FULL', current_timestamp()),
(4, 'primary_key', 'branch_code', current_timestamp()),

-- ================= CREDIT BUREAU REPORTS =================
(5, 'load_type', 'MERGE', current_timestamp()),
(5, 'primary_key', 'customer_id', current_timestamp()),
(5, 'watermark_column', 'bureau_pull_date', current_timestamp()),

-- ================= PAYMENT GATEWAY LOGS =================
(6, 'load_type', 'APPEND', current_timestamp()),
(6, 'primary_key', 'txn_id', current_timestamp()),
(6, 'watermark_column', 'processed_timestamp', current_timestamp());

In [0]:
INSERT INTO banking.metadata.tables
VALUES (
    7,
    'customer_360',
    'silver',
    NULL,
    NULL,
    NULL,
    'gold',
    NULL,
    NULL,
    'gold',
    TRUE,
    1,
    current_timestamp()
);

-- COMMAND ----------

INSERT INTO banking.metadata.tables
VALUES
(
8,
'branch_performance',
'silver',
NULL,
NULL,
NULL,
'gold',
NULL,
NULL,
'gold',
TRUE,
2,
current_timestamp()
),

(
9,
'transaction_channel_summary',
'silver',
NULL,
NULL,
NULL,
'gold',
NULL,
NULL,
'gold',
TRUE,
3,
current_timestamp()
),

(
10,
'daily_bank_kpi',
'silver',
NULL,
NULL,
NULL,
'gold',
NULL,
NULL,
'gold',
TRUE,
4,
current_timestamp()
);

-- COMMAND ----------

INSERT INTO banking.metadata.tables
VALUES (
    11,
    'risk_customer_summary',
    'silver',
    NULL,
    NULL,
    NULL,
    'gold',
    NULL,
    NULL,
    'gold',
    TRUE,
    1,
    current_timestamp()
);

In [0]:
delete from banking.metadata.tables where table_id = 11

In [0]:
INSERT INTO banking.metadata.table_watermarks VALUES
(1, '1900-01-01 00:00:00', current_timestamp(), NULL),
(2, '1900-01-01 00:00:00', current_timestamp(), NULL),
(3, '1900-01-01 00:00:00', current_timestamp(), NULL),
(5, '1900-01-01 00:00:00', current_timestamp(), NULL),
(6, '1900-01-01 00:00:00', current_timestamp(), NULL);

In [0]:
TRUNCATE TABLE banking.metadata.table_watermarks